# Janus — train your model on a free Colab GPU

Train a new **Janus** model version without a local GPU. This notebook uses a
free Colab **T4** to LoRA-fine-tune the base Gemma weights on transcripts of
Janus's own successful runs, then exports a **GGUF** you load back into Ollama.

**Before you start (on your own machine):**
```bash
janus dataset          # builds datasets/sft.jsonl from your winning runs
```
Then upload that `sft.jsonl` when this notebook asks for it.

**Runtime → Change runtime type → T4 GPU** before running the cells.


## 1. Check the GPU


In [ ]:
!nvidia-smi


## 2. Install Unsloth
Unsloth makes Gemma LoRA training fit comfortably on a free T4 and can export GGUF directly.


In [ ]:
%%capture
!pip install unsloth


## 3. Upload your training data
Upload the `sft.jsonl` produced by `janus dataset` on your machine.


In [ ]:
from google.colab import files
uploaded = files.upload()   # choose sft.jsonl
DATA_PATH = next(iter(uploaded))
print('using', DATA_PATH)


## 4. Configuration
Set `BASE_MODEL` to the **full-precision** Hugging Face repo of your base model.
It must be the trainable base of the GGUF you run in Ollama — you cannot train
the `IQ3_M` GGUF itself. The default below is Gemma 3n E2B; change it to match
your model (e.g. your own uncensored base repo on Hugging Face).


In [ ]:
BASE_MODEL   = 'unsloth/gemma-3n-E2B-it'   # <-- set to YOUR base model repo
MAX_SEQ_LEN  = 2048
LORA_R       = 16
LORA_ALPHA   = 32
EPOCHS       = 1
LR           = 2e-4
QUANT        = 'q4_k_m'                     # GGUF quantization
OUT_TAG_HINT = 'janus'                      # just a filename hint


## 5. Load the base model + attach LoRA


In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = BASE_MODEL,
    max_seq_length = MAX_SEQ_LEN,
    load_in_4bit = True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_R,
    lora_alpha = LORA_ALPHA,
    lora_dropout = 0.0,
    target_modules = ['q_proj','k_proj','v_proj','o_proj',
                      'gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing = 'unsloth',
)


## 6. Format the dataset with the Gemma chat template
Each line is `{"messages": [...]}` — the same trajectories Janus mined from its
winning runs. We render them with the tokenizer's chat template and train on them.


In [ ]:
from datasets import load_dataset

ds = load_dataset('json', data_files=DATA_PATH, split='train')

def _format(ex):
    return {'text': tokenizer.apply_chat_template(ex['messages'], tokenize=False)}

ds = ds.map(_format, remove_columns=ds.column_names)
print('examples:', len(ds))
print(ds[0]['text'][:500])


## 7. Train


In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = ds,
    args = SFTConfig(
        dataset_text_field = 'text',
        max_seq_length = MAX_SEQ_LEN,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = EPOCHS,
        learning_rate = LR,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 5,
        optim = 'adamw_8bit',
        output_dir = 'outputs',
    ),
)
trainer.train()


## 8. Export to GGUF
Merges the LoRA into the base weights and writes a quantized GGUF Ollama can run.


In [ ]:
model.save_pretrained_gguf('janus_gguf', tokenizer, quantization_method=QUANT)

import glob, os
gguf = glob.glob('janus_gguf/*.gguf')[0]
print('GGUF:', gguf, round(os.path.getsize(gguf)/1e6, 1), 'MB')


## 9. Package and download
Bundles the GGUF with a ready-made Ollama `Modelfile`.


In [ ]:
modelfile = 'janus_gguf/Modelfile'
with open(modelfile, 'w') as fh:
    fh.write(f'FROM {os.path.basename(gguf)}\nPARAMETER temperature 0.7\n')

!cd janus_gguf && zip -r /content/janus_model.zip *.gguf Modelfile
from google.colab import files
files.download('/content/janus_model.zip')


## 10. Load it back into Janus (on your machine)
Unzip the download, then let Janus register it into the model lineage:
```bash
unzip janus_model.zip -d janus_model
janus model import janus_model/*.gguf --adopt   # registers + activates janus:vN
janus model list                                 # confirm the new version is adopted
```
`janus model import` creates the Ollama tag, records it in the lineage (parent,
base, scorecard slot), and — with `--adopt` — makes it the active model. Prefer
to A/B it first? Import without `--adopt`, then `janus bench` each tag and
`janus model use janus:vN` the winner.

That's the full no-local-GPU loop: **`janus dataset` → Colab → `janus model import`.**
